# EDA: Oxford-IIIT Pet

Короткий разведочный анализ: размер выборок, баланс классов, примеры картинок и их размеры.
Ноутбук работает и в Colab, и локально (первая ячейка подготовит данные).

In [ ]:
# Общая подготовка: Google Drive, репозиторий, данные.
# В Colab: Среда выполнения -> Сменить среду выполнения -> T4 GPU.
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/semenovfilipp/image-classification.git"
BRANCH = "feature/image-classification"  # после merge в main поменять на "main"

REPO_DIR = "/content/image-classification" if IN_COLAB else os.getcwd()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/oxford_pets"   # данные и результаты живут здесь
    os.makedirs(DRIVE, exist_ok=True)
    if not os.path.exists("/content/image-classification"):
        !git clone -b {BRANCH} {REPO_URL} /content/image-classification
    %cd /content/image-classification
    !git pull -q
    !pip install -q -r requirements.txt
else:
    DRIVE = "data"                                 # локальный запуск из notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        %cd ..
    REPO_DIR = os.getcwd()

OUT = f"{DRIVE}/outputs"
ARCHIVE = f"{DRIVE}/oxford_pets_processed.tar"
os.makedirs(OUT, exist_ok=True)

# Данные: при первом запуске скачиваем и уменьшаем (~5 мин), архив кладём на Drive.
# В следующих сессиях просто распаковываем архив (~30 сек) на быстрый локальный диск Colab.
if os.path.exists("data/processed/oxford_pets/train.csv"):
    print("Данные уже подготовлены")
elif os.path.exists(ARCHIVE):
    !mkdir -p data/processed && tar -xf "{ARCHIVE}" -C data/processed
else:
    !python scripts/prepare_data.py
    if IN_COLAB:
        !tar -cf "{ARCHIVE}" -C data/processed oxford_pets
!ls data/processed/oxford_pets

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path("data/processed/oxford_pets")
classes = json.load(open(ROOT / "classes.json"))
splits = {s: pd.read_csv(ROOT / f"{s}.csv") for s in ("train", "val", "test")}
# в Oxford-IIIT Pet 12 пород кошек и 25 пород собак
CATS = {"Abyssinian", "Bengal", "Birman", "Bombay", "British Shorthair", "Egyptian Mau",
        "Maine Coon", "Persian", "Ragdoll", "Russian Blue", "Siamese", "Sphynx"}
for s, df in splits.items():
    df["species"] = np.where(df["class_name"].isin(CATS), "cat", "dog")
    print(f"{s:>5}: {len(df):>5} картинок, {df['label'].nunique()} классов")

## Баланс классов

In [ ]:
counts = pd.DataFrame({s: df["class_name"].value_counts() for s, df in splits.items()}).fillna(0).astype(int)
counts = counts.sort_values("train")
ax = counts.plot.barh(stacked=True, figsize=(9, 11))
ax.set_xlabel("число картинок"); ax.set_title("Картинок на класс")
plt.tight_layout(); plt.show()
print(counts["train"].describe()[["min", "mean", "max"]])
print(pd.concat(splits)["species"].value_counts())

## Примеры: по одной картинке на класс

In [ ]:
df = splits["train"].groupby("label").sample(1, random_state=0).sort_values("label")
fig, axes = plt.subplots(5, 8, figsize=(18, 12))
for ax in axes.flat:
    ax.axis("off")
for ax, (_, r) in zip(axes.flat, df.iterrows()):
    ax.imshow(Image.open(ROOT / "images" / r["filename"]))
    ax.set_title(r["class_name"], fontsize=9, color="tab:orange" if r["species"] == "cat" else "tab:blue")
plt.suptitle("оранжевые — кошки, синие — собаки"); plt.tight_layout(); plt.show()

## Похожие породы: несколько примеров двух классов, которые легко спутать

In [ ]:
pair = ["American Pit Bull Terrier", "Staffordshire Bull Terrier"]
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for row, cls in zip(axes, pair):
    sample = splits["train"][splits["train"]["class_name"] == cls].sample(6, random_state=1)
    for ax, fn in zip(row, sample["filename"]):
        ax.imshow(Image.open(ROOT / "images" / fn)); ax.axis("off"); ax.set_title(cls, fontsize=8)
plt.tight_layout(); plt.show()

## Размеры картинок
Берём исходные картинки, если они остались после `prepare_data.py`; иначе уменьшенные
(у них короткая сторона 256, но соотношение сторон то же).

In [ ]:
raw_dir = Path("data/raw/oxford-iiit-pet/images")
src_dir = raw_dir if raw_dir.exists() else ROOT / "images"
files = sorted(src_dir.glob("*.jpg"))[::5]  # каждая 5-я — для скорости
sizes = pd.DataFrame([Image.open(f).size for f in files], columns=["w", "h"])
sizes["aspect"] = sizes["w"] / sizes["h"]
print(f"источник: {src_dir}, картинок в выборке: {len(sizes)}")
print(sizes.describe().round(2))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(sizes["w"], sizes["h"], s=4, alpha=0.4); axes[0].set_xlabel("ширина"); axes[0].set_ylabel("высота")
axes[1].hist(sizes["aspect"], bins=50); axes[1].set_xlabel("ширина / высота")
plt.tight_layout(); plt.show()

## Выводы
Проверьте цифры по выходам ячеек выше.

* 37 классов (12 пород кошек, 25 пород собак), около 7.4k картинок; официальный test (~3.7k) оставлен
  как есть, trainval поделён на train/val 85/15 стратифицированно.
* Классы почти сбалансированы (около 100 картинок на класс в trainval), поэтому accuracy
  подходит как основная метрика; macro-F1 считаем дополнительно.
* Данных на класс мало (~85 в train), так что обучение с нуля будет сильно переобучаться,
  и стоит ждать большого выигрыша от transfer learning с ImageNet.
* Картинки разного размера и соотношения сторон, животное обычно в центре: используем
  `RandomResizedCrop` + flip для train и `Resize` + `CenterCrop` для val/test.
* Задача fine-grained: некоторые породы очень похожи (например, питбуль и стаффорд),
  именно они будут давать основные ошибки.